In [ ]:
import json
import re
from pathlib import Path
from collections import defaultdict

In [ ]:
labels = ["C-EXP", "MC-EXP", "P-EXP"]


In [ ]:
def sort_spans(spans):
    return sorted(spans, key=lambda x: x[1][0])

In [ ]:
def jsonl_to_list(fname):
    labels_per_instance = defaultdict(list)
    with open(fname, 'r') as f:
        for i, line in enumerate(f):
            instance = json.loads(line)
            instance_id = instance['id']
            labels = instance['labels']

            instance_labels = []
            for label in labels:
                start = label['start']
                end = label['end']

                instance_labels.append((label['label'], [start, end]))

            instance_labels = sort_spans(instance_labels)
            labels_per_instance[instance_id] = instance_labels
    return labels_per_instance

In [ ]:
def label_freq(annotations, label):
    all_annotations = []
    for isinstance_id, pred_labels in annotations.items():
        for x in pred_labels:
            all_annotations.append(x[0])

    label_freq = sum([1 for a in all_annotations if a == label])

    return label_freq

In [ ]:
def span_intersection(gold_span, pred_span):
    x = range(gold_span[0], gold_span[1])
    y = range(pred_span[0], pred_span[1])
    inter = set(x).intersection(y)
    return len(inter)

In [ ]:
def score_all(prec_numerator, prec_denominator, rec_numerator, rec_denominator):
    prec, rec, f1 = (0, 0, 0)
    if prec_denominator > 0:
        prec = prec_numerator / prec_denominator
    if rec_denominator > 0:
        rec = rec_numerator / rec_denominator
    if prec_denominator == 0 and rec_denominator == 0:
        f1 = 1.0
    if prec > 0 and rec > 0:
        f1 = 2 * (prec * rec / (prec + rec))

    return prec, rec, f1

In [ ]:
def score_per_span(gold_annotation, pred_annotation):
    prec_cnt = sum([len(pred_annotation[x]) for x in pred_annotation])
    rec_cnt = sum([len(gold_annotation[x]) for x in gold_annotation])

    label_prec = {label: 0 for label in labels}
    label_rec = {label: 0 for label in labels}
    cum_prec, cum_rec = (0, 0)
    f1_instances = []

    for instance_id, instance_pred in pred_annotation.items():
        instance_gold = gold_annotation[instance_id]

        par_cum_prec, par_cum_rec = (0,0)
        for i, pred_label in enumerate(instance_pred):
            pred_length = pred_label[1][1] - pred_label[1][0]

            for j, gold_label in enumerate(instance_gold):
                if pred_label[0] == gold_label[0]:
                    intersection = span_intersection(gold_label[1], pred_label[1])
                    gold_length = gold_label[1][1] - gold_label[1][0]
                    span_prec = intersection / pred_length
                    par_cum_prec += span_prec
                    cum_prec += span_prec
                    label_prec[gold_label[0]] += span_prec

                    span_rec = intersection / gold_length
                    par_cum_rec += span_rec
                    cum_rec += span_rec
                    label_rec[gold_label[0]] += span_rec

        par_prec, par_rec, par_f1 = score_all(par_cum_prec, len(instance_pred), par_cum_rec, len(instance_gold))
        f1_instances.append(par_f1)

    prec, rec, f1 = score_all(cum_prec, prec_cnt, cum_rec, rec_cnt)

    f1_per_label = []
    prec_per_label = []
    rec_per_label = []
    for label in label_prec.keys():
        label_p, label_r, label_f1 = score_all(label_prec[label], label_freq(pred_annotation, label), label_rec[label], label_freq(gold_annotation, label))
        f1_per_label.append(label_f1)
        prec_per_label.append(label_p)
        rec_per_label.append(label_r)
    return prec, rec, f1, f1_per_label, prec_per_label, rec_per_label

In [ ]:
def evaluate(gold_annotations, pred_annotations, per_label=True):
    precision, recall, f1, f1_per_label, prec_per_label, rec_per_label = score_per_span(gold_annotations, pred_annotations)
    if per_label:
        print(f"F1= {f1} | Precision= {precision} | Recall= {recall}")
        for label, prec, rec, f1 in zip(labels, prec_per_label, rec_per_label, f1_per_label):
            print(f"{label}: Precision= {prec}, Recall= {rec}, F1= {f1}")
    else:
        print(f"Micro-F1= {f1} | Macro-f1= {sum(f1_per_label)/len(f1_per_label)} | Precision= {precision} | Recall= {recall}")

In [ ]:
import json
import re
from pathlib import Path

In [ ]:
def safe_div(num, den):
    return num / den if den else 0

def prf(tp, fp, fn):
    precision = safe_div(tp, tp + fp)
    recall = safe_div(tp, tp + fn)
    f1 = safe_div(2 * precision * recall, precision + recall)
    return precision, recall, f1

def spans_overlap(span1, span2):
    return max(span1[0], span2[0]) < min(span1[1], span2[1])

def span_iou(span1, span2):
    inter = max(0, min(span1[1], span2[1]) - max(span1[0], span2[0]))
    union = max(span1[1], span2[1]) - min(span1[0], span2[0])
    return safe_div(inter, union)

In [ ]:
def evaluate_overlap_f1(gold_annotations, pred_annotations, iou_threshold=0.1, label_sensitive=True):
    tp = fp = fn = 0

    for doc_id in gold_annotations:
        gold = gold_annotations[doc_id]
        pred = pred_annotations.get(doc_id, [])

        matched_gold = set()

        for pred_ann in pred:
            pred_label, pred_span = pred_ann

            best_match = None
            best_iou = 0

            for g_idx, gold_ann in enumerate(gold):

                if g_idx in matched_gold:
                    continue

                gold_label, gold_span = gold_ann

                if label_sensitive and pred_label != gold_label:
                    continue

                iou = span_iou(gold_span, pred_span)

                if iou >= iou_threshold and iou > best_iou:
                    best_iou = iou
                    best_match = g_idx

            if best_match is not None:
                tp += 1
                matched_gold.add(best_match)
            else:
                fp += 1

        fn += len(gold) - len(matched_gold)

    return prf(tp, fp, fn)

In [ ]:
def evaluate_token_level_f1(gold_annotations, pred_annotations, label_sensitive=True):

    gold_tokens = set()
    pred_tokens = set()

    for doc_id, anns in gold_annotations.items():

        for label, span in anns:

            for char_pos in range(span[0], span[1]):

                item = (doc_id, char_pos, label) if label_sensitive else (doc_id, char_pos)

                gold_tokens.add(item)

    for doc_id, anns in pred_annotations.items():

        for label, span in anns:

            for char_pos in range(span[0], span[1]):

                item = (doc_id, char_pos, label) if label_sensitive else (doc_id, char_pos)

                pred_tokens.add(item)

    tp = len(gold_tokens & pred_tokens)
    fp = len(pred_tokens - gold_tokens)
    fn = len(gold_tokens - pred_tokens)

    return prf(tp, fp, fn)

In [ ]:
def get_sentence_spans(text):

    sentence_spans = []

    pattern = re.compile(r'[^.!?]+[.!?]*', re.MULTILINE)

    for match in pattern.finditer(text):

        start, end = match.span()

        sentence = text[start:end].strip()

        if sentence:
            sentence_spans.append((start, end))

    return sentence_spans


def span_to_sentence_ids(span, sentence_spans):

    sent_ids = set()

    for i, sent_span in enumerate(sentence_spans):

        if spans_overlap(span, sent_span):
            sent_ids.add(i)

    return sent_ids

In [ ]:
def evaluate_sentence_level_f1(gold_annotations, pred_annotations, raw_text_root, label_sensitive=True):

    gold_sentence_labels = set()
    pred_sentence_labels = set()

    all_doc_ids = set(gold_annotations.keys()) | set(pred_annotations.keys())

    for doc_id in all_doc_ids:

        with open(raw_text_root / f"{doc_id}.txt", "r", encoding="utf-8") as f:
            text = f.read()

        sentence_spans = get_sentence_spans(text)

        for label, span in gold_annotations.get(doc_id, []):

            for sent_id in span_to_sentence_ids(span, sentence_spans):

                item = (doc_id, sent_id, label) if label_sensitive else (doc_id, sent_id)

                gold_sentence_labels.add(item)

        for label, span in pred_annotations.get(doc_id, []):

            for sent_id in span_to_sentence_ids(span, sentence_spans):

                item = (doc_id, sent_id, label) if label_sensitive else (doc_id, sent_id)

                pred_sentence_labels.add(item)

    tp = len(gold_sentence_labels & pred_sentence_labels)
    fp = len(pred_sentence_labels - gold_sentence_labels)
    fn = len(gold_sentence_labels - pred_sentence_labels)

    return prf(tp, fp, fn)

In [ ]:
def evaluate_all_metrics(gold_annotations, pred_annotations, raw_text_root):

    print("\n===== OVERLAP F1 =====")

    p, r, f1 = evaluate_overlap_f1(
        gold_annotations,
        pred_annotations,
        iou_threshold=0.1,
        label_sensitive=True
    )

    print(f"Precision={p:.4f}")
    print(f"Recall={r:.4f}")
    print(f"F1={f1:.4f}")

    print("\n===== CHARACTER-OFFSET F1 =====")

    p, r, f1 = evaluate_character_offset_f1(
        gold_annotations,
        pred_annotations,
        label_sensitive=True
    )

    print(f"Precision={p:.4f}")
    print(f"Recall={r:.4f}")
    print(f"F1={f1:.4f}")

    print("\n===== SENTENCE-LEVEL F1 =====")

    p, r, f1 = evaluate_sentence_level_f1(
        gold_annotations,
        pred_annotations,
        raw_text_root,
        label_sensitive=True
    )

    print(f"Precision={p:.4f}")
    print(f"Recall={r:.4f}")
    print(f"F1={f1:.4f}")

# ARGUMENTATION LAYER

In [ ]:
project_root = Path(r"C:\Users\Luiza\OneDrive\Desktop\MA THESIS\FINAL\FINAL_ARG_prompting")

raw_text_root = project_root / "raw_texts"
annotation_root = project_root / "annotations"
llm_output_root = project_root / "llm_outputs"
eval_input_root = project_root / "eval_inputs"

eval_input_root.mkdir(exist_ok=True)

# Human INCEpTION JSON parser

In [ ]:
def parse_inception_json(filepath):
    with open(filepath, "r", encoding="utf-8") as f:
        data = json.load(f)

    feature_structures = data["%FEATURE_STRUCTURES"]

    name_to_ui = {}

    for fs in feature_structures:
        if fs.get("%TYPE") == "de.tudarmstadt.ukp.clarin.webanno.api.type.FeatureDefinition":
            if fs.get("name") and fs.get("uiName"):
                name_to_ui[fs["name"]] = fs["uiName"]

    ignore_features = {"IClaimtext", "MCIMPtext", "PIMPtext"}

    annotation_layers = {
        "webanno.custom.Argumentation",
        #"webanno.custom.Narration"
    }

    annotations = []

    for fs in feature_structures:
        if fs.get("%TYPE") not in annotation_layers:
            continue

        start = fs.get("begin")
        end = fs.get("end")

        if start is None or end is None:
            continue

        for feature_name, value in fs.items():
            if feature_name.startswith("%") or feature_name.startswith("@"):
                continue
            if feature_name in {"begin", "end"}:
                continue
            if feature_name in ignore_features:
                continue

            if value is True:
                annotations.append({
                    "label": name_to_ui.get(feature_name, feature_name),
                    "start": int(start),
                    "end": int(end)
                })

    return annotations

# LLM output offsets

In [ ]:
def read_llm_output(filepath):
    with open(filepath, "r", encoding="utf-8") as f:
        output = f.read().strip()

    output = re.sub(r"```json", "", output, flags=re.IGNORECASE)
    output = re.sub(r"```", "", output)

    start = output.find("{")
    end = output.rfind("}")

    if start == -1 or end == -1:
        raise ValueError(f"No JSON object found in {filepath}")

    data = json.loads(output[start:end+1])

    if "annotations" in data:
        return data["annotations"]

    if "argumentation" in data or "narration" in data:
        return data.get("argumentation", []) + data.get("narration", [])

    raise ValueError("Unknown LLM output format")

In [ ]:
def llm_text_to_offsets(llm_annotations, ted_text):
    converted = []

    for ann in llm_annotations:
        # Accept both possible keys:
        # normal format: {"label": "...", "text": "..."}
        # odd format:    {"label": "...", "span": "..."}
        span_text = ann.get("text") or ann.get("span")
        label = ann.get("label")

        # Skip malformed annotations
        if not span_text or not label:
            continue

        start = ted_text.find(span_text)

        if start == -1:
            continue

        end = start + len(span_text)

        converted.append({
            "label": label,
            "start": start,
            "end": end
        })

    return converted

In [ ]:
def write_jsonl(filepath, documents):
    with open(filepath, "w", encoding="utf-8") as f:
        for document_id, annotations in documents.items():
            row = {
                "id": document_id,
                "labels": annotations
            }
            f.write(json.dumps(row, ensure_ascii=False) + "\n")

In [ ]:
def filter_annotations(annotations, selected_labels):
    return [
        {
            "label": ann["label"],
            "start": ann["start"],
            "end": ann["end"]
        }
        for ann in annotations
        if ann["label"] in selected_labels
    ]

In [ ]:
def main(reference_file, pred_file, raw_text_root=None, show_extra_metrics=True):
    reference_annotations = jsonl_to_list(reference_file)
    pred_annotations = jsonl_to_list(pred_file)

    print("\n===== RELAXED-F1 =====")
    evaluate(reference_annotations, pred_annotations, True)

    if show_extra_metrics and raw_text_root is not None:
        evaluate_all_metrics(
            reference_annotations,
            pred_annotations,
            raw_text_root
        )

In [ ]:
def run_all(task_type, model_name, pv, annotator, labels_list, docs):
    document_ids = docs
    prompt_version = pv
    annotator_file = f"{annotator}.json"

    reference_documents = {}
    pred_documents = {}

    for document_id in document_ids:
        with open(raw_text_root / f"{document_id}.txt", "r", encoding="utf-8") as f:
            ted_text = f.read()

        human_annotations = parse_inception_json(
            annotation_root / document_id / annotator_file
        )

        llm_annotations_raw = read_llm_output(
            llm_output_root / f"{document_id}_{model_name}_prompt-{prompt_version}_{task_type}.json"
        )

        llm_annotations = llm_text_to_offsets(llm_annotations_raw, ted_text)

        reference_documents[document_id] = filter_annotations(human_annotations, labels_list)
        pred_documents[document_id] = filter_annotations(llm_annotations, labels_list)

    reference_file = eval_input_root / f"reference_{annotator}_{task_type}.jsonl"
    pred_file = eval_input_root / f"pred_{model_name}_prompt-{prompt_version}_{task_type}.jsonl"

    write_jsonl(reference_file, reference_documents)
    write_jsonl(pred_file, pred_documents)

    print(f"\n\n==============================")
    print(f"LLM: {model_name} | Prompt: {prompt_version} | Reference annotator: {annotator}")
    print(f"==============================")

    main(reference_file, pred_file, raw_text_root)

In [ ]:
task_type = "argumentation"
model_name = "mistral-small-3.2-24b-instruct"
prompt_version = "final"
labels = ["C-EXP", "MC-EXP", "P-EXP"]
docs = ["TED_001", "TED_002", "TED_004", "TED_005", "TED_006", "TED_007", "TED_008", "TED_010", "TED_011", "TED_012", "TED_015", "TED_016", "TED_017", "TED_018", "TED_019", "TED_020"]

In [ ]:
run_all(task_type, model_name, prompt_version, "filip", labels, docs )

In [ ]:
run_all(task_type, model_name, prompt_version, "nabhani", labels, docs )